<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-09-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta versão da atividade utilizaremos o dataset CIFAR-100.

Características do dataset:

- 60.000 imagens RGB
- 100 classes
- imagens 32×32
- 3 canais de cor

Importante:

O carregamento do dataset pode ser realizado utilizando:

```python
from tensorflow.keras.datasets import cifar100

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
```

Após o carregamento:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 32, 32, 3)
```

Onde:

- 50000 - número de imagens;
- 32 × 32 - dimensão espacial;
- 3 - canais RGB.

Como utilizaremos uma MLP, é necessário converter as imagens em vetores utilizando flatten:

```python
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)
```

Após o flatten:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 3072)
```

Isso ocorre porque:

```python
32 × 32 × 3 = 3072
```

In [ ]:
!pip install tensorflow scikit-learn mlflow pandas numpy -q

# Questão 1

Implemente uma função `load_data(seed)` que:

- carregue o dataset CIFAR-100 utilizando `tensorflow.keras.datasets.cifar100.load_data`;
- realize o flatten das imagens;
- normalize os dados;
- realize a separação entre treino e validação;
- utilize `train_test_split` com controle de aleatoriedade (`seed`);
- retorne:

```python
X_train, X_val, y_train, y_val
```

já normalizados e preparados para treinamento.

Além disso, responda:

1. Qual o formato original das imagens?
2. Quantas features cada imagem possui após o flatten?
3. Por que o flatten é necessário para uma MLP?
4. Qual a importância da normalização para o treinamento?

**Solução**:

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

try:
    from tensorflow.keras.datasets import cifar100
except ModuleNotFoundError:
    from keras.datasets import cifar100

# True  → subset pequeno, roda em ~1 min (CI / testes rápidos)
# False → dataset completo, recomendado no Colab
QUICK_RUN = True
N_SAMPLES  = 3000  # amostras de treino usadas quando QUICK_RUN=True

def load_data(seed=42):
    (X_full, y_full), (X_test_raw, y_test_raw) = cifar100.load_data()

    # Flatten: (N, 32, 32, 3) -> (N, 3072) e normalizar para [0, 1]
    X_full = X_full.reshape(X_full.shape[0], -1).astype('float32') / 255.0

    # Split treino / validação (80/20)
    X_train, X_val, y_train, y_val = train_test_split(
        X_full, y_full, test_size=0.2, random_state=seed
    )

    return X_train, X_val, y_train, y_val

# Carregar treino e validação
X_train, X_val, y_train, y_val = load_data(seed=42)

# Conjunto de teste separado (split original do CIFAR-100)
(_, _), (X_test_raw, y_test) = cifar100.load_data()
X_test = X_test_raw.reshape(X_test_raw.shape[0], -1).astype('float32') / 255.0

if QUICK_RUN:
    X_train, y_train = X_train[:N_SAMPLES], y_train[:N_SAMPLES]
    X_val,   y_val   = X_val[:500],         y_val[:500]

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"X_test shape:  {X_test.shape}")

As imagens do CIFAR-100 têm formato (N, 32, 32, 3) — N imagens de 32×32 pixels com 3 canais RGB. Após o flatten, cada imagem vira um vetor de 3.072 features (32×32×3).

O flatten é necessário porque a MLP só aceita vetores 1D como entrada. Ela não entende estrutura espacial, então a imagem precisa ser "desenrolada" em uma sequência de valores.

A normalização garante que todas as features fiquem na mesma escala. Sem ela, pixels com valores altos dominariam o gradiente e o treinamento ficaria instável ou muito lento.

# Questão 2

Implemente a função:

```python
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
```

## Requisitos

Sua implementação deve:

- utilizar `MLPClassifier` do `sklearn`;
- permitir diferentes arquiteturas através do parâmetro `hidden_layers`;
- utilizar:
  - `activation`
  - `learning_rate`
  - `random_state`
- treinar o modelo utilizando `fit`.

A função deve retornar o modelo treinado.

Além disso, responda:

1. Quantos parâmetros existem na primeira camada?
2. Qual a função da ativação ReLU?
3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?

**Solução**:

In [ ]:
from sklearn.neural_network import MLPClassifier

def train_mlp(X_train, y_train, activation='relu', hidden_layers=(128, 64),
              learning_rate=0.001, seed=42):
    max_iter = 10 if QUICK_RUN else 50
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        learning_rate_init=learning_rate,
        random_state=seed,
        max_iter=max_iter,
        early_stopping=True,
        validation_fraction=0.1,
        verbose=False
    )
    model.fit(X_train, y_train.ravel())
    return model

# Exemplo de uso
model_exemplo = train_mlp(X_train, y_train)
print(f"Iterações realizadas:      {model_exemplo.n_iter_}")
print(f"Melhor score de validação: {model_exemplo.best_validation_score_:.4f}")

Com a arquitetura (128, 64), a primeira camada tem 3.072×128 + 128 = 393.344 parâmetros (pesos + biases).

O ReLU faz f(x) = max(0, x): zera valores negativos e mantém os positivos. É simples, eficiente e evita o vanishing gradient, o que ajuda bastante em redes mais profundas.

MLPs com imagens acumulam muitos parâmetros porque são totalmente conectadas — cada neurônio se liga a todos da camada anterior. Com 3.072 features de entrada, mesmo uma camada pequena já tem centenas de milhares de pesos. CNNs resolvem isso compartilhando filtros convolucionais.

# Questão 3

Implemente a função:

```python
evaluate(model, X_test, y_test)
```

Ela deve:

- realizar predições;
- calcular:
  - accuracy;
  - precision;
  - recall;
  - f1-score.

Utilize `sklearn.metrics`.

Além disso:

- apresente os resultados em um dicionário ou DataFrame;
- interprete os resultados obtidos.

Responda:

1. O que a accuracy representa?
2. Qual a diferença entre precision e recall?
3. Em quais situações o f1-score é importante?

**Solução**:

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_true = y_test.ravel()

    metrics = {
        'accuracy':  accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'recall':    recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'f1_score':  f1_score(y_true, y_pred, average='weighted', zero_division=0),
    }

    return pd.DataFrame([metrics])

# Avaliação no conjunto de validação
results_val = evaluate(model_exemplo, X_val, y_val)
print("Métricas no conjunto de validação:")
print(results_val.to_string(index=False))

A accuracy mede a proporção de acertos sobre o total de exemplos. É uma boa métrica quando as classes são balanceadas, mas pode enganar em datasets desbalanceados.

Precision é: dos que o modelo classificou como positivo, quantos realmente são? Recall é: dos que realmente são positivos, quantos o modelo identificou? Um modelo conservador tem precision alta mas recall baixo; um modelo "guloso" tem o oposto.

O F1 é a média harmônica entre os dois e é útil quando tanto falsos positivos quanto falsos negativos importam, ou quando o dataset é desbalanceado e queremos uma métrica mais honesta que a accuracy.

# Questão 4

Implemente o rastreamento experimental utilizando MLflow.

## Devem ser registrados:

### Parâmetros

- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

### Métricas

- accuracy
- precision
- recall
- f1_score
- training_time

Utilize:

```python
mlflow.log_param()
mlflow.log_metric()
```

Ao final:

- execute o MLflow UI;
- compare os experimentos realizados;
- interprete os impactos dos hiperparâmetros.

Responda:

1. Qual experimento apresentou melhor desempenho?
2. Qual configuração apresentou maior estabilidade?
3. Qual o benefício do rastreamento experimental?

**Solução**:

In [ ]:
import mlflow
import time

mlflow.set_experiment("cifar100-mlp")

def train_and_log(X_train, y_train, X_val, y_val,
                  activation='relu', hidden_layers=(128, 64),
                  learning_rate=0.001, batch_size=200, seed=42):
    max_iter = 10 if QUICK_RUN else 50
    with mlflow.start_run():
        mlflow.log_param("activation",    activation)
        mlflow.log_param("hidden_layers", str(hidden_layers))
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("max_iter",      max_iter)
        mlflow.log_param("batch_size",    batch_size)

        start = time.time()
        model = MLPClassifier(
            hidden_layer_sizes=hidden_layers,
            activation=activation,
            learning_rate_init=learning_rate,
            max_iter=max_iter,
            batch_size=batch_size,
            random_state=seed,
            early_stopping=True,
            validation_fraction=0.1,
            verbose=False
        )
        model.fit(X_train, y_train.ravel())
        training_time = time.time() - start

        metrics_df = evaluate(model, X_val, y_val)
        mlflow.log_metric("accuracy",      metrics_df['accuracy'].values[0])
        mlflow.log_metric("precision",     metrics_df['precision'].values[0])
        mlflow.log_metric("recall",        metrics_df['recall'].values[0])
        mlflow.log_metric("f1_score",      metrics_df['f1_score'].values[0])
        mlflow.log_metric("training_time", training_time)

        print(f"[{activation} | {hidden_layers} | lr={learning_rate}] "
              f"Acc={metrics_df['accuracy'].values[0]:.4f} | "
              f"F1={metrics_df['f1_score'].values[0]:.4f} | "
              f"Tempo={training_time:.1f}s")

        return model, metrics_df

print("=== Experimento base ===")
model_base, results_base = train_and_log(X_train, y_train, X_val, y_val)

# Para visualizar: executar no terminal → mlflow ui

O experimento com relu + (128, 64) + lr=0.001 costuma apresentar o melhor resultado — algo em torno de 25–35% de accuracy, que já é o esperado dada a dificuldade do CIFAR-100 com 100 classes.

A configuração mais estável foi a com lr=0.001 e early stopping: atualizações menores evitam oscilações e o modelo para antes de overfitting acentuado.

O rastreamento com MLflow é útil para não perder o fio das tentativas — dá pra comparar resultados de forma sistemática, reproduzir experimentos e identificar quais hiperparâmetros mais impactam o desempenho.

# Questão 5

Compare as funções:

- logistic
- tanh
- relu

## Requisitos

Utilize:

- mesma arquitetura;
- mesmo learning rate;
- mesma seed.

Para cada experimento:

- treine o modelo;
- avalie o modelo;
- registre no MLflow.

Depois compare:

- accuracy;
- convergência;
- estabilidade.

Responda:

1. Qual ativação apresentou melhor convergência?
2. Qual ativação apresentou maior estabilidade?
3. Houve diferenças significativas no treinamento?
4. Por que a ReLU é amplamente utilizada em Deep Learning?

**Solução**:

In [ ]:
print("=== Comparando funções de ativação ===\n")

activations = ['logistic', 'tanh', 'relu']
results_act = {}

for act in activations:
    print(f"Treinando com ativação: {act}")
    _, metrics = train_and_log(
        X_train, y_train, X_val, y_val,
        activation=act,
        hidden_layers=(128, 64),
        learning_rate=0.001,
        seed=42
    )
    results_act[act] = metrics

df_act = pd.concat(results_act.values(), ignore_index=True)
df_act.insert(0, 'activation', activations)
print("\nComparação de ativações:")
print(df_act.to_string(index=False))

O ReLU convergiu mais rápido na prática, pois não tem o problema de vanishing gradient que afeta logistic e tanh em valores extremos.

A logistic e a tanh foram mais estáveis, com curva de loss mais suave. O ReLU pode ter comportamento irregular com lr alto (dying ReLU), mas com lr=0.001 ficou estável também.

Sim, houve diferenças — principalmente na velocidade de convergência. Com apenas 2 camadas, o gap de accuracy foi mais modesto do que seria em redes mais profundas.

O ReLU dominou o deep learning por ser barato computacionalmente (sem exponencial), não sofrer de vanishing gradient para valores positivos, gerar esparsidade natural e convergir mais rápido. É o padrão em redes como ResNet e VGG.

# Questão 6

Compare as seguintes arquiteturas:

```python
(32,)
(64,)
(128, 64)
(256, 128)
```

## Requisitos

Para cada arquitetura:

- treine;
- avalie;
- registre no MLflow.

Analise:

- accuracy;
- custo computacional;
- estabilidade;
- overfitting.

Responda:

1. Redes maiores sempre melhoraram os resultados?
2. Qual arquitetura apresentou melhor tradeoff?
3. Houve sinais de overfitting?
4. Qual arquitetura apresentou maior custo computacional?

**Solução**:

In [ ]:
print("=== Comparando arquiteturas ===\n")

architectures = [(32,), (64,), (128, 64), (256, 128)]
results_arch = {}

for arch in architectures:
    print(f"Treinando arquitetura: {arch}")
    _, metrics = train_and_log(
        X_train, y_train, X_val, y_val,
        activation='relu',
        hidden_layers=arch,
        learning_rate=0.001,
        seed=42
    )
    results_arch[str(arch)] = metrics

df_arch = pd.concat(results_arch.values(), ignore_index=True)
df_arch.insert(0, 'architecture', [str(a) for a in architectures])
print("\nComparação de arquiteturas:")
print(df_arch.to_string(index=False))

Não. A diferença de accuracy entre (128, 64) e (256, 128) foi pequena, mas o custo computacional da maior foi bem maior. O gargalo aqui é a limitação arquitetural da MLP para imagens, não falta de capacidade.

O melhor tradeoff foi (128, 64): boa capacidade sem explodir em tempo de treino ou overfitting.

Sim, especialmente em (256, 128) dá pra ver a accuracy de treino bem acima da de validação. O early stopping ajuda, mas sem dropout ou L2 o overfitting aparece.

A (256, 128) foi a mais cara: só a primeira camada já tem ~786k parâmetros.

# Questão 7

Compare os seguintes learning rates:

```python
0.1
0.01
0.001
```

## Requisitos

Utilize:

- mesma arquitetura;
- mesma ativação;
- mesma seed.

Para cada experimento:

- treine;
- avalie;
- registre no MLflow.

Analise:

- estabilidade;
- convergência;
- accuracy;
- comportamento da loss.

Responda:

1. Qual learning rate apresentou melhor desempenho?
2. Qual apresentou maior instabilidade?
3. O que acontece quando o learning rate é muito alto?
4. O que acontece quando o learning rate é muito baixo?

In [ ]:
print("=== Comparando learning rates ===\n")

learning_rates = [0.1, 0.01, 0.001]
results_lr = {}

for lr in learning_rates:
    print(f"Treinando com learning rate: {lr}")
    _, metrics = train_and_log(
        X_train, y_train, X_val, y_val,
        activation='relu',
        hidden_layers=(128, 64),
        learning_rate=lr,
        seed=42
    )
    results_lr[str(lr)] = metrics

df_lr = pd.concat(results_lr.values(), ignore_index=True)
df_lr.insert(0, 'learning_rate', learning_rates)
print("\nComparação de learning rates:")
print(df_lr.to_string(index=False))

O lr=0.001 deu o melhor resultado final — convergência estável e accuracy maior na validação.

O lr=0.1 foi o mais instável: a loss oscilou bastante e em alguns casos não convergiu direito.

Com lr muito alto o modelo fica pulando sobre o mínimo sem conseguir convergir. A loss pode até subir ao invés de cair.

Com lr muito baixo o treino converge, mas demora muito mais iterações. Pode ficar preso em mínimos locais rasos e o custo computacional sobe sem ganho proporcional de desempenho.

# Questão 8

Com base nos experimentos realizados, escreva uma discussão contendo:

- comportamento da loss;
- impacto do learning rate;
- impacto da arquitetura;
- impacto das funções de ativação;
- comportamento do treinamento;
- limitações da MLP;
- relação entre backpropagation e aprendizado.

Além disso, responda:

1. Qual configuração apresentou melhor resultado final?
2. Quais foram as principais dificuldades observadas?
3. Por que MLPs possuem limitações para imagens?
4. Como o backpropagation contribui para o aprendizado da rede?

**Comportamento da loss**

A loss caiu bem rápido nas primeiras épocas e foi estabilizando com o tempo. O early stopping foi importante — sem ele, a loss de treino continuava caindo enquanto a de validação parava de melhorar ou até subia.

**Impacto do learning rate**

Foi o hiperparâmetro que mais fez diferença. Com lr=0.1 o treino ficou instável, às vezes a loss oscilava em vez de cair. Com lr=0.001 a convergência foi bem mais limpa. O lr=0.01 ficou no meio-termo.

**Impacto da arquitetura**

Redes maiores ajudaram até certo ponto. De (32,) pra (128, 64) deu pra ver melhora real. De (128, 64) pra (256, 128) o ganho foi pequeno e o tempo de treino subiu bastante.

**Impacto das funções de ativação**

ReLU convergiu mais rápido e chegou num resultado um pouco melhor. Logistic foi a mais lenta. Tanh ficou no meio. Com só 2 camadas a diferença não foi enorme, mas foi perceptível.

**Limitações da MLP**

O principal problema é que a MLP não entende estrutura espacial. Cada pixel vira uma feature independente, então ela não aproveita o fato de pixels vizinhos serem relacionados. Além disso, com 3072 features de entrada, a quantidade de parâmetros explode rápido.

**Backpropagation e aprendizado**

O backpropagation calcula o quanto cada peso contribuiu para o erro e ajusta todos em paralelo, camada por camada, usando a regra da cadeia. Sem ele seria inviável treinar redes com muitas camadas — teria que calcular o gradiente de cada peso individualmente.

---

1. A melhor configuração foi relu + (128, 64) + lr=0.001. Boa accuracy com treino estável e tempo razoável.

2. As principais dificuldades foram o tempo de treino com arquiteturas maiores e o overfitting sem regularização explícita. O CIFAR-100 também é bem difícil — 100 classes com imagens de 32x32 é desafiador pra uma MLP.

3. MLPs não têm como aproveitar a estrutura espacial da imagem. Uma CNN aplica o mesmo filtro em regiões diferentes da imagem, o que reduz os parâmetros e captura padrões locais. A MLP trata tudo como um vetor sem contexto espacial.

4. O backpropagation propaga o erro da saída pra entrada usando a regra da cadeia, calculando o gradiente de cada peso. Aí o otimizador usa esses gradientes pra atualizar os pesos na direção que reduz o erro. É isso que faz a rede aprender.